In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    when,
    trim,
    to_timestamp,
    regexp_replace,
    min as spark_min,
    max as spark_max,
    sum as spark_sum,
    year,
    month
)

SOURCE_FILE = "physical_vendas_caixa.csv"
SOURCE_PATH = f"{RAW_BATCH_PATH}{SOURCE_FILE}"

CSV_OPTIONS = {
    "header": "true",
    "inferSchema": "false"
}

EXPECTED_COLUMNS = [
    "id_transacao",
    "id_loja",
    "id_caixa",
    "id_operador",
    "dt_venda",
    "valor_total_venda",
    "cpf_cliente",
    "tipo_pagamento"
]

adls_options = get_adls_options()

df_raw = read_source_csv(
    spark=spark,
    source_path=SOURCE_PATH,
    adls_options=adls_options,
    csv_options=CSV_OPTIONS
)

print(f"Arquivo lido: {SOURCE_PATH}")
print(f"Total de linhas: {df_raw.count()}")
print(f"Total de colunas: {len(df_raw.columns)}")

df_raw.printSchema()
display(df_raw.limit(10))

In [0]:
actual_columns = df_raw.columns

missing_columns = [c for c in EXPECTED_COLUMNS if c not in actual_columns]
extra_columns = [c for c in actual_columns if c not in EXPECTED_COLUMNS]

print(f"Colunas esperadas: {len(EXPECTED_COLUMNS)}")
print(f"Colunas encontradas: {len(actual_columns)}")
print(f"Colunas ausentes: {missing_columns}")
print(f"Colunas extras: {extra_columns}")

if missing_columns:
    raise Exception(f"Arquivo Raw com colunas obrigatórias ausentes: {missing_columns}")

df_nulls = df_raw.select([
    count(
        when(
            col(c).isNull() | (trim(col(c)) == ""),
            True
        )
    ).alias(c)
    for c in EXPECTED_COLUMNS
])

display(df_nulls)

total_linhas = df_raw.count()
ids_distintos = df_raw.select("id_transacao").distinct().count()
ids_duplicados = total_linhas - ids_distintos

print(f"Total de linhas: {total_linhas}")
print(f"IDs distintos: {ids_distintos}")
print(f"IDs duplicados: {ids_duplicados}")

In [0]:
df_analise = (
    df_raw
    .withColumn("dt_venda_ts", to_timestamp(col("dt_venda")))
    .withColumn(
        "valor_total_venda_decimal",
        regexp_replace(col("valor_total_venda"), ",", ".").cast("decimal(18,2)")
    )
)

df_resumo = df_analise.select(
    count("*").alias("total_linhas"),
    countDistinct("id_transacao").alias("qtd_transacoes_distintas"),
    spark_min("dt_venda_ts").alias("data_venda_mais_antiga"),
    spark_max("dt_venda_ts").alias("data_venda_mais_recente"),
    count(when(col("dt_venda_ts").isNull(), True)).alias("dt_venda_invalida_ou_nula"),
    count(when(col("valor_total_venda_decimal").isNull(), True)).alias("valor_total_venda_invalido_ou_nulo"),
    count(when(col("valor_total_venda_decimal") <= 0, True)).alias("valor_total_venda_menor_ou_igual_zero"),
    count(when(~trim(col("tipo_pagamento")).isin("Cartão", "Dinheiro", "Pix"), True)).alias("tipo_pagamento_fora_do_esperado")
)

display(df_resumo)

display(
    df_analise
    .groupBy("tipo_pagamento")
    .agg(
        count("*").alias("qtd_transacoes"),
        spark_sum("valor_total_venda_decimal").alias("receita_total")
    )
    .orderBy("tipo_pagamento")
)

display(
    df_analise
    .groupBy(
        year("dt_venda_ts").alias("ano_venda"),
        month("dt_venda_ts").alias("mes_venda")
    )
    .agg(
        count("*").alias("qtd_transacoes"),
        spark_sum("valor_total_venda_decimal").alias("receita_total")
    )
    .orderBy("ano_venda", "mes_venda")
)

In [0]:
BACKUP_FILE = "physical_vendas_caixa_backup_20260618_201815.csv"
BACKUP_PATH = f"{RAW_BATCH_PATH}{BACKUP_FILE}"

df_backup = read_source_csv(
    spark=spark,
    source_path=BACKUP_PATH,
    adls_options=adls_options,
    csv_options=CSV_OPTIONS
)

print(f"Arquivo backup lido: {BACKUP_PATH}")
print(f"Total de linhas backup: {df_backup.count()}")
print(f"Total de colunas backup: {len(df_backup.columns)}")

df_backup.printSchema()
display(df_backup.limit(10))

In [0]:
colunas_atual = df_raw.columns
colunas_backup = df_backup.columns

print(f"Colunas iguais: {colunas_atual == colunas_backup}")
print(f"Colunas atual: {colunas_atual}")
print(f"Colunas backup: {colunas_backup}")

linhas_atual = df_raw.count()
linhas_backup = df_backup.count()

ids_atual = df_raw.select("id_transacao").distinct().count()
ids_backup = df_backup.select("id_transacao").distinct().count()

print(f"Linhas atual: {linhas_atual}")
print(f"Linhas backup: {linhas_backup}")
print(f"Diferença atual - backup: {linhas_atual - linhas_backup}")

print(f"IDs distintos atual: {ids_atual}")
print(f"IDs distintos backup: {ids_backup}")
print(f"Diferença IDs atual - backup: {ids_atual - ids_backup}")

In [0]:
df_atual_tratado = (
    df_raw
    .withColumn("dt_venda_ts", to_timestamp(col("dt_venda")))
    .withColumn(
        "valor_total_venda_decimal",
        regexp_replace(col("valor_total_venda"), ",", ".").cast("decimal(18,2)")
    )
)

df_backup_tratado = (
    df_backup
    .withColumn("dt_venda_ts", to_timestamp(col("dt_venda")))
    .withColumn(
        "valor_total_venda_decimal",
        regexp_replace(col("valor_total_venda"), ",", ".").cast("decimal(18,2)")
    )
)

df_comparativo_periodo = (
    df_atual_tratado.select(
        count("*").alias("linhas_atual"),
        countDistinct("id_transacao").alias("ids_atual"),
        spark_min("dt_venda_ts").alias("data_min_atual"),
        spark_max("dt_venda_ts").alias("data_max_atual"),
        spark_sum("valor_total_venda_decimal").alias("receita_atual")
    )
    .crossJoin(
        df_backup_tratado.select(
            count("*").alias("linhas_backup"),
            countDistinct("id_transacao").alias("ids_backup"),
            spark_min("dt_venda_ts").alias("data_min_backup"),
            spark_max("dt_venda_ts").alias("data_max_backup"),
            spark_sum("valor_total_venda_decimal").alias("receita_backup")
        )
    )
)

display(df_comparativo_periodo)

In [0]:
ids_somente_atual = (
    df_raw
    .select("id_transacao")
    .join(
        df_backup.select("id_transacao"),
        on="id_transacao",
        how="left_anti"
    )
    .count()
)

ids_somente_backup = (
    df_backup
    .select("id_transacao")
    .join(
        df_raw.select("id_transacao"),
        on="id_transacao",
        how="left_anti"
    )
    .count()
)

print(f"IDs que existem no arquivo atual, mas não no backup: {ids_somente_atual}")
print(f"IDs que existem no backup, mas não no arquivo atual: {ids_somente_backup}")

In [0]:
df_atual_pagamento = (
    df_atual_tratado
    .groupBy("tipo_pagamento")
    .agg(
        count("*").alias("qtd_atual"),
        spark_sum("valor_total_venda_decimal").alias("receita_atual")
    )
)

df_backup_pagamento = (
    df_backup_tratado
    .groupBy("tipo_pagamento")
    .agg(
        count("*").alias("qtd_backup"),
        spark_sum("valor_total_venda_decimal").alias("receita_backup")
    )
)

df_comparativo_pagamento = (
    df_atual_pagamento
    .join(df_backup_pagamento, on="tipo_pagamento", how="full")
    .fillna(0)
    .withColumn("diferenca_qtd_atual_menos_backup", col("qtd_atual") - col("qtd_backup"))
    .withColumn("diferenca_receita_atual_menos_backup", col("receita_atual") - col("receita_backup"))
    .orderBy("tipo_pagamento")
)

display(df_comparativo_pagamento)

In [0]:
df_novos = (
    df_atual_tratado
    .join(
        df_backup.select("id_transacao"),
        on="id_transacao",
        how="left_anti"
    )
)

display(
    df_novos.select(
        count("*").alias("qtd_registros_novos"),
        countDistinct("id_transacao").alias("ids_novos_distintos"),
        spark_min("dt_venda_ts").alias("data_min_novos"),
        spark_max("dt_venda_ts").alias("data_max_novos"),
        spark_sum("valor_total_venda_decimal").alias("receita_nova")
    )
)

In [0]:
display(
    df_novos
    .groupBy(
        year("dt_venda_ts").alias("ano_venda"),
        month("dt_venda_ts").alias("mes_venda")
    )
    .agg(
        count("*").alias("qtd_transacoes_novas"),
        spark_sum("valor_total_venda_decimal").alias("receita_nova")
    )
    .orderBy("ano_venda", "mes_venda")
)

In [0]:
df_cpf_atual = df_raw.select(
    count("*").alias("linhas_atual"),
    count(when(col("cpf_cliente").isNull() | (trim(col("cpf_cliente")) == ""), True)).alias("cpf_nulo_atual")
)

df_cpf_backup = df_backup.select(
    count("*").alias("linhas_backup"),
    count(when(col("cpf_cliente").isNull() | (trim(col("cpf_cliente")) == ""), True)).alias("cpf_nulo_backup")
)

df_cpf_novos = df_novos.select(
    count("*").alias("linhas_novas"),
    count(when(col("cpf_cliente").isNull() | (trim(col("cpf_cliente")) == ""), True)).alias("cpf_nulo_novos")
)

display(
    df_cpf_atual
    .crossJoin(df_cpf_backup)
    .crossJoin(df_cpf_novos)
)